# Hamiltonians Simulation with Tierkreis and Guppy

This exercise simulates a situation in which we want to upgrade an exisiting project by migrating from `pytket` to `guppy`.
Further, we know that the workflow will need parallelization capabilites in the future and is part of a larger workflow.
To make the project ready for such a scenario we are going to use `tierkreis`.

**Learning outcomes**
1. Understanding python workers in tierkreis
2. Levaraging automatic parallelism in tierkreis
3. Writing composed guppy code

Look to the code and fix the todos.

## Optional Task 1: The worker
Implement the guppy worker.
In tierkreis, workers are standalone projects to prevent dependency issues.
Hence, the worker is defined in a separate file.
[impl.py](../workers/new_worker/tkr_new_worker_impl/impl.py)

**Hint: Use the `hugr.package.Package` type as serialization format.**

### Solution

The solution can be found in the reference worker [impl.py](../workers/reference_worker/tkr_refence_worker_impl/impl.py)


## Task 2: Updating the exisiting graph

The exisiting graphs uses `pytket.Circuit` as ansatz.
We now need to update the workflow to use `hugr.Package` instead.
First we update the `exp_val` subgraph:
1. Update the parameter types
2. Use the apropriate worker functions
The typehints here can help you identify the solution.

**Info:**
The solution uses the `reference_worker`.
If you have implememented the new worker yourself, replace all occurences of `reference_worker` with `new_worker`.

In [ ]:
# This cell is only necessary if you run this from the VSCODE notebook
import sys
from pathlib import Path

_tkr_dir = Path().resolve().parents[2]
if str(_tkr_dir) not in sys.path:
    sys.path.insert(0, str(_tkr_dir))

In [ ]:
from typing import  NamedTuple

from guppy_worker import emulate, to_backend_result
from reference_worker import append_pauli_measurements, optimise_ansatz
from pytket_worker import expectation
from tierkreis.builder import Graph
from tierkreis.models import TKR, OpaqueType, Workflow

class SubmitInputs(NamedTuple):
    circuit: TKR[OpaqueType["hugr.package.Package"]]
    pauli_string: TKR[OpaqueType["pytket._tket.pauli.QubitPauliString"]]
    n_shots: TKR[int]


def exp_val() -> Workflow[SubmitInputs, TKR[float]]:
    g = Graph(SubmitInputs, TKR[float])

    circuit = g.inputs.circuit
    pauli_string = g.inputs.pauli_string
    n_shots = g.inputs.n_shots

    measurement_circuit = g.task(append_pauli_measurements(circuit, pauli_string))
    compiled_circuit = g.task(optimise_ansatz(measurement_circuit))
    result = g.task(emulate(compiled_circuit, g.const(4), n_shots))
    backend_result = g.task(to_backend_result(result))
    av = g.task(expectation(backend_result))
    return g.finish_with_outputs(av)

Next we're going to upgrade the Hamiltonian Simulation:

In [ ]:
from tierkreis.builtins import tkr_zip, unzip
from tkr.graphs.guppy_main import FoldGraphInputs, compute_terms, fold_graph
from reference_worker import substitute

class SymbolicExecutionInputs(NamedTuple):
    a: TKR[float]
    b: TKR[float]
    c: TKR[float]
    ham: TKR[list[tuple[OpaqueType["pytket._tket.pauli.QubitPauliString"], float]]]
    ansatz: TKR[OpaqueType["hugr.package.Package"]]

def hamiltonian_sim() -> Workflow[SymbolicExecutionInputs, TKR[float]]:
    simulation_graph = Graph(SymbolicExecutionInputs, TKR[float])
    substituted_ansatz = simulation_graph.task(
        substitute(
            simulation_graph.inputs.ansatz,
            simulation_graph.inputs.a,
            simulation_graph.inputs.b,
            simulation_graph.inputs.c,
        )
    )
    pauli_strings_list, parameters_list = simulation_graph.task(
        unzip(simulation_graph.inputs.ham)
    )
    input_circuits = simulation_graph.map(
        lambda x: SubmitInputs(substituted_ansatz, x, simulation_graph.const(100)),
        pauli_strings_list,
    )
    exp_values = simulation_graph.map(exp_val(), input_circuits)
    tuple_values = simulation_graph.task(tkr_zip(exp_values, parameters_list))
    fold_inputs = FoldGraphInputs(simulation_graph.const(0.0), tuple_values)
    computed = simulation_graph.eval(fold_graph(compute_terms()), fold_inputs)
    return simulation_graph.finish_with_outputs(computed)

We now define our inputs:

In [ ]:
from typing import Any

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array
from guppylang.std.quantum import cx, qubit, rz
from pytket import Qubit
from pytket.pauli import Pauli, QubitPauliString

@guppy(link_name="ansatz")
def ansatz(qbts: array[qubit, 4], a: float, b: float, c: float) -> None:  # type: ignore
    for i in array(0, 1, 2):
        cx(qbts[i], qbts[i + 1])
    rz(qbts[0], angle(a))
    for i in array(2, 1, 0):
        cx(qbts[i], qbts[i + 1])
    rz(qbts[0], angle(b))
    for i in array(0, 1, 2):
        cx(qbts[i], qbts[i + 1])
    rz(qbts[0], angle(c))
    for i in array(2, 1, 0):
        cx(qbts[i], qbts[i + 1])

def inputs() -> dict[str, Any]:
    ansatz_package = guppy.library(ansatz).compile()
    qubits = [Qubit(0), Qubit(1), Qubit(2), Qubit(3)]
    hamiltonian = [
        (QubitPauliString(qubits, [Pauli.X, Pauli.Y, Pauli.X, Pauli.I]).to_list(), 0.1),
        (QubitPauliString(qubits, [Pauli.Y, Pauli.Z, Pauli.X, Pauli.Z]).to_list(), 0.5),
        (QubitPauliString(qubits, [Pauli.X, Pauli.Y, Pauli.Z, Pauli.I]).to_list(), 0.3),
        (QubitPauliString(qubits, [Pauli.Z, Pauli.Y, Pauli.X, Pauli.Y]).to_list(), 0.6),
    ]
    return {
        "ansatz": ansatz_package,
        "a": 0.2,
        "b": 0.55,
        "c": 0.75,
        "ham": hamiltonian,
    }


Finally we can run the graph:

In [ ]:
import logging
from uuid import UUID

from tierkreis.controller import run_graph
from tierkreis.executor import ShellExecutor
from tierkreis.storage import FileStorage, read_outputs


def main() -> None:
    graph = hamiltonian_sim()
    storage = FileStorage(
        workflow_id=UUID(int=12345), name="guppy_hamiltonian_simulation"
    )
    executor = ShellExecutor(None, storage.workflow_dir)
    storage.clean_graph_files()
    run_graph(storage, executor, graph, inputs())
    # storage = debug_graph(graph, inputs()) # TODO use this instead of run_graph to debug the graph
    result = read_outputs(graph, storage)
    print("Value is: ", result)

logging.basicConfig(level=logging.INFO)
main()